# Chhaap: Warli LoRA training on a free Colab T4

1. Runtime > Change runtime type > T4 GPU.
2. `warli_train.zip` (45 cleaned 512x512 Warli images) is already in the project folder. To rebuild it: `cd data/clean/warli && zip -r ../../../warli_train.zip train && cd -`
3. Run the cells below in order. The upload cell asks for `warli_train.zip`.
4. The last cell downloads `warli_lora_outputs.zip`. Unzip it into the project's `models/` folder. It holds `warli_lora.safetensors`, `checkpoints/warli_lora_{500,1000,1500}.safetensors` and `train_loss.csv`. The app loads the LoRA automatically on its next start.

Captions are made here with `Salesforce/blip-image-captioning-base` (the laptop's connection is too slow for the 1 GB model) and every caption starts with the trigger word `warli_style`.

**Mandana LoRA (v2):** set `STYLE = "mandana"` in the upload cell and upload `mandana_train.zip` (built by `scripts/prepare_mandana.py`). Everything else is the same. Unzip `mandana_lora_outputs.zip` into `models/`; its logs carry a `mandana_` prefix so the Warli logs stay intact.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone --depth 1 https://github.com/huggingface/diffusers
!pip -q install ./diffusers
!pip -q install -r diffusers/examples/text_to_image/requirements.txt bitsandbytes tensorboard
!pip -q uninstall -y torchao  # Colab preinstalls torchao 0.10; current peft refuses to import with torchao < 0.16, and training does not use it

In [ ]:
STYLE = "warli"  # "mandana" trains the Mandana LoRA from mandana_train.zip (trigger word mandana_style)
LOG = "" if STYLE == "warli" else f"{STYLE}_"  # Mandana logs never overwrite the Warli ones in models/
from google.colab import files
up = files.upload()  # choose {STYLE}_train.zip
!rm -rf data && mkdir data && unzip -q {STYLE}_train.zip -d data && ls data/train | wc -l

In [ ]:
import json, torch
from pathlib import Path
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
proc = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base", torch_dtype=torch.float16).to("cuda")
rows = []
for p in sorted(Path("data/train").glob("*.png")):
    x = proc(images=Image.open(p).convert("RGB"), return_tensors="pt").to("cuda", torch.float16)
    cap = proc.decode(blip.generate(**x, max_new_tokens=30)[0], skip_special_tokens=True)
    rows.append({"file_name": p.name, "text": f"{STYLE}_style, {cap}"})
Path("data/train/metadata.jsonl").write_text("\n".join(json.dumps(r) for r in rows))
print(len(rows), rows[:3])
del blip; torch.cuda.empty_cache()

In [ ]:
!accelerate launch --mixed_precision=fp16 diffusers/examples/text_to_image/train_text_to_image_lora.py \
  --pretrained_model_name_or_path=stable-diffusion-v1-5/stable-diffusion-v1-5 \
  --train_data_dir=data/train --caption_column=text --resolution=512 --random_flip \
  --train_batch_size=1 --gradient_accumulation_steps=1 --max_train_steps=1500 \
  --learning_rate=1e-4 --lr_scheduler=constant --lr_warmup_steps=0 --rank=16 \
  --mixed_precision=fp16 --use_8bit_adam --checkpointing_steps=500 --seed=0 \
  --output_dir=lora_out --report_to=tensorboard --logging_dir=logs 2>&1 | tee train_log.txt | grep -v it/s | tail -20

In [ ]:
import csv, glob, shutil, os
assert os.path.exists("lora_out/pytorch_lora_weights.safetensors"), "Training did not finish: read train_log.txt (tail) before exporting"
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
os.makedirs("out/checkpoints", exist_ok=True)
ev = EventAccumulator(glob.glob("lora_out/logs/**/events.out.tfevents.*", recursive=True)[0]); ev.Reload()
tag = [t for t in ev.Tags()["scalars"] if "loss" in t][0]
with open(f"out/{LOG}train_loss.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["step", "loss"]); w.writerows((e.step, e.value) for e in ev.Scalars(tag))
for s in (500, 1000, 1500):
    src = f"lora_out/checkpoint-{s}/pytorch_lora_weights.safetensors"
    if os.path.exists(src): shutil.copy(src, f"out/checkpoints/{STYLE}_lora_{s}.safetensors")
shutil.copy("lora_out/pytorch_lora_weights.safetensors", f"out/{STYLE}_lora.safetensors")
shutil.copy("data/train/metadata.jsonl", f"out/{LOG}captions.jsonl")
shutil.copy("train_log.txt", f"out/{LOG}train_log.txt")
!cd out && zip -qr ../{STYLE}_lora_outputs.zip . && ls -R
files.download(f"{STYLE}_lora_outputs.zip")